## 1) โหลด speech VAD (จาก CSV ที่เราเพิ่งสร้าง)

In [1]:
import pandas as pd
import numpy as np

speech_csv = "emotion_results_wavlm_all.csv"
df_speech = pd.read_csv(speech_csv)

df_speech

,filename,dialogue_id,utterance_id,arousal,valence,dominance
0,dialogue_2_utterance_1.wav,2,1,0.577664,0.162604,0.648455
1,dialogue_2_utterance_2.wav,2,2,0.459763,0.500791,0.508309
2,dialogue_2_utterance_3.wav,2,3,0.867881,0.767458,0.873423
3,dialogue_2_utterance_4.wav,2,4,0.379486,0.537600,0.474338


## 2) โหลด text VAD

In [3]:
text_csv = "../../own_script/dialogue_2/dialogue_2_vad_text.csv"
df_text = pd.read_csv(text_csv)

df_text


,utterance_id,text,valence_text,arousal_text,dominance_text,valence_text_n,arousal_text_n,dominance_text_n
0,1,Why are you bothering me? What's the problem?,2.515900,3.496335,3.119745,-0.242050,0.248168,0.059873
1,2,"Ahh that thing again, can you just stay away f...",2.503102,3.572586,3.061257,-0.248449,0.286293,0.030629
2,3,I'm fine! I am very good and doing well at the...,4.039449,3.855152,3.643599,0.519725,0.427576,0.321799
3,4,"Besides, you are the one who seems to be doing...",3.347648,3.262022,3.264729,0.173824,0.131011,0.132364


## 3) สเกล speech logits ไปช่วง [-1,1] --- Comment ที่จะใช้

### สเกลโดยการใช้ mapping คงที่ เพื่อคงค่า absoloute ไว้

In [4]:
cols = ["arousal", "dominance", "valence"]

for c in cols:
    x = df_speech[c].values  # สมมติค่านี้ ~ [0, 1] อยู่แล้ว
    x = np.clip(x, 0.0, 1.0)  # กันกรณีหลุดนิดหน่อย
    df_speech[c + "_scaled"] = 2 * x - 1   # map 0..1 -> -1..1

df_speech[["arousal_scaled", "dominance_scaled", "valence_scaled"]].describe()


,arousal_scaled,dominance_scaled,valence_scaled
count,4.000000,4.000000,4.000000
mean,0.142397,0.252262,-0.015774
std,0.427759,0.362537,0.498712
min,-0.241028,-0.051325,-0.674792
25%,-0.120612,-0.000368,-0.167512
50%,0.037427,0.156764,0.038391
75%,0.300437,0.409394,0.190129
max,0.735761,0.746846,0.534915


#### Check preserved order after min/max scaling for speech_df

In [5]:
cols_speech = ["arousal", "dominance", "valence"]

def check_order_preserved(df, col, scaled_col):
    idx = np.argsort(df[col].values)
    x = df[col].values[idx]
    y = df[scaled_col].values[idx]
    return np.all(y[1:] >= y[:-1])

print("=== speech ===")
for c in cols_speech:
    ok = check_order_preserved(df_speech, c, c + "_scaled")
    print(f"{c}: order preserved? {ok}")

=== speech ===
arousal: order preserved? True
dominance: order preserved? True
valence: order preserved? True


In [6]:
df_speech

,filename,dialogue_id,utterance_id,arousal,valence,dominance,arousal_scaled,dominance_scaled,valence_scaled
0,dialogue_2_utterance_1.wav,2,1,0.577664,0.162604,0.648455,0.155329,0.296909,-0.674792
1,dialogue_2_utterance_2.wav,2,2,0.459763,0.500791,0.508309,-0.080474,0.016618,0.001581
2,dialogue_2_utterance_3.wav,2,3,0.867881,0.767458,0.873423,0.735761,0.746846,0.534915
3,dialogue_2_utterance_4.wav,2,4,0.379486,0.537600,0.474338,-0.241028,-0.051325,0.075200


## 4) สเกล text VAD ไปช่วง [-1,1]

### สเกลโดยการใช้ mapping คงที่ เพื่อคงค่า absoloute ไว้

In [7]:
cols_text = ["valence_text", "arousal_text", "dominance_text"]

for c in cols_text:
    x = df_text[c].values          # raw ในช่วง ~[1, 5]
    x = np.clip(x, 1.0, 5.0)       # กันหลุดนอกสเกลเล็กน้อย
    x_01 = (x - 1.0) / (5.0 - 1.0) # map 1..5 -> 0..1
    df_text[c.replace("_text", "_scaled")] = x_01 * 2 - 1  # 0..1 -> -1..1



df_text[["valence_scaled", "arousal_scaled", "dominance_scaled"]].describe()


,valence_scaled,arousal_scaled,dominance_scaled
count,4.000000,4.000000,4.000000
mean,0.050762,0.273262,0.136166
std,0.369836,0.122267,0.130936
min,-0.248449,0.131011,0.030629
25%,-0.243650,0.218878,0.052562
50%,-0.034113,0.267230,0.096118
75%,0.260299,0.321614,0.179723
max,0.519725,0.427576,0.321799


#### Check preserved order after min/max scaling

In [8]:
import numpy as np

cols_text = ["valence_text", "arousal_text", "dominance_text"]

def check_order_preserved(df, col, scaled_col):
    # sort ตามค่าดั้งเดิม
    idx = np.argsort(df[col].values)
    x = df[col].values[idx]
    y = df[scaled_col].values[idx]

    # x ต้อง non-decreasing อยู่แล้วเพราะ sort; เช็คว่า y ก็ non-decreasing
    return np.all(y[1:] >= y[:-1])

print("=== text ===")
for c in cols_text:
    base_col   = c
    scaled_col = c.replace("_text", "_scaled")
    ok = check_order_preserved(df_text, base_col, scaled_col)
    print(f"{base_col}: order preserved? {ok}")


=== text ===
valence_text: order preserved? True
arousal_text: order preserved? True
dominance_text: order preserved? True


## 5) ตั้งชื่อให้ไม่ชนกัน แล้ว merge

In [9]:
import pandas as pd
import numpy as np

# 1) เตรียมให้ utterance_id ตรงกัน (dialogue 1)
speech = df_speech[df_speech["dialogue_id"] == 2].sort_values("utterance_id").reset_index(drop=True)
text   = df_text.sort_values("utterance_id").reset_index(drop=True)

assert (speech["utterance_id"].to_numpy() == text["utterance_id"].to_numpy()).all()

# 2) ดึงเฉพาะ valence + arousal ที่สเกลแล้ว [-1,1]
df_dissonance = pd.DataFrame({
    "utterance_id": speech["utterance_id"],
    "aro_s": speech["arousal_scaled"].to_numpy(),
    "val_s": speech["valence_scaled"].to_numpy(),
    "aro_t": text["arousal_scaled"].to_numpy(),
    "val_t": text["valence_scaled"].to_numpy(),
})


## 6) คำนวณ dissonance (เวอร์ชันที่กันบั๊ก dtype แล้ว)

In [10]:
# 3) delta ต่อมิติ
df_dissonance["delta_arousal"] = (df_dissonance["aro_s"] - df_dissonance["aro_t"]).abs()
df_dissonance["delta_valence"] = (df_dissonance["val_s"] - df_dissonance["val_t"]).abs()

# 4) threshold บน [-1,1]
ARO_THR = 0.5
VAL_THR = 0.5

df_dissonance["dissonant_arousal"] = df_dissonance["delta_arousal"] > ARO_THR
df_dissonance["dissonant_valence"] = df_dissonance["delta_valence"] > VAL_THR
df_dissonance["dissonant_any"]     = (
    df_dissonance["dissonant_arousal"] | df_dissonance["dissonant_valence"]
)

# # 5) optional: score เดียว (L2 จาก 2 มิติ)
# df_dissonance["dissonance_l2"] = np.sqrt(
#     df_dissonance["delta_arousal"]**2 + df_dissonance["delta_valence"]**2
# )

In [11]:
df_dissonance

,utterance_id,aro_s,val_s,aro_t,val_t,delta_arousal,delta_valence,dissonant_arousal,dissonant_valence,dissonant_any
0,1,0.155329,-0.674792,0.248168,-0.242050,0.092839,0.432742,False,False,False
1,2,-0.080474,0.001581,0.286293,-0.248449,0.366767,0.250030,False,False,False
2,3,0.735761,0.534915,0.427576,0.519725,0.308185,0.015191,False,False,False
3,4,-0.241028,0.075200,0.131011,0.173824,0.372038,0.098624,False,False,False


In [12]:
df_speech

,filename,dialogue_id,utterance_id,arousal,valence,dominance,arousal_scaled,dominance_scaled,valence_scaled
0,dialogue_2_utterance_1.wav,2,1,0.577664,0.162604,0.648455,0.155329,0.296909,-0.674792
1,dialogue_2_utterance_2.wav,2,2,0.459763,0.500791,0.508309,-0.080474,0.016618,0.001581
2,dialogue_2_utterance_3.wav,2,3,0.867881,0.767458,0.873423,0.735761,0.746846,0.534915
3,dialogue_2_utterance_4.wav,2,4,0.379486,0.537600,0.474338,-0.241028,-0.051325,0.075200


In [13]:
df_text

,utterance_id,text,valence_text,arousal_text,dominance_text,valence_text_n,arousal_text_n,dominance_text_n,valence_scaled,arousal_scaled,dominance_scaled
0,1,Why are you bothering me? What's the problem?,2.515900,3.496335,3.119745,-0.242050,0.248168,0.059873,-0.242050,0.248168,0.059872
1,2,"Ahh that thing again, can you just stay away f...",2.503102,3.572586,3.061257,-0.248449,0.286293,0.030629,-0.248449,0.286293,0.030629
2,3,I'm fine! I am very good and doing well at the...,4.039449,3.855152,3.643599,0.519725,0.427576,0.321799,0.519725,0.427576,0.321799
3,4,"Besides, you are the one who seems to be doing...",3.347648,3.262022,3.264729,0.173824,0.131011,0.132364,0.173824,0.131011,0.132364


In [15]:
output_path = "../../own_script/dialogue_2/dialogue_2_dissonance.csv"
df_dissonance.to_csv(output_path, index=False)
print("Saved:", output_path)


Saved: ../../own_script/dialogue_2/dialogue_2_dissonance.csv
